# Module 7 - Activity 4: Eli5, LIME and Mlxtend

**Brief:** run the three packages - **Eli5** ("explain like I'm 5"), **LIME** (local surrogate) and
**Mlxtend** (decision boundaries). Inspect feature importances, interpret predictions, look at
decision regions. Then: *how do these compare with the explainer dashboard?*

**The question worth actually asking.** Running three tools that each print a bar chart is busywork.
The real experiment is to point **all of them at the same prediction** and check whether they
**agree**. Because if two explanations of one decision contradict each other, at most one is right -
and nothing on the screen tells you which.

> That is Aha's *"a wrong explanation is worse than none"* (Res. 4) turned into an experiment.

We reuse the **wine** data (Assessment 2) and one random forest throughout, so every difference we
see comes from the **explainer**, not the model.

**Spoiler, so you know what you are looking for:** they do not agree. One of these libraries hands us
a fluent, well-formatted explanation with **every sign backwards** - it reports that a wine's vinegar
taint is what made it *good*. We catch it not with a better chart, but by going back and
**intervening on the model itself**. That habit is the actual deliverable of this notebook.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from scipy.stats import spearmanr

SEED = 42
np.random.seed(SEED)
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)

WINE = Path('../../../assignments/Assessment2/dataset/winequality-red.csv')
df = pd.read_csv(WINE, sep=';')
df.columns = [c.strip().replace(' ', '_') for c in df.columns]
df['good'] = (df['quality'] >= 7).astype(int)

X = df.drop(columns=['quality', 'good'])
y = df['good']
FEATS = list(X.columns)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25,
                                          random_state=SEED, stratify=y)
model = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
print(f'accuracy {accuracy_score(y_te, model.predict(X_te)):.3f} | '
      f'roc_auc {roc_auc_score(y_te, model.predict_proba(X_te)[:,1]):.3f}')
print('One model. Four explainers. Do they tell the same story?')

accuracy 0.940 | roc_auc 0.952
One model. Four explainers. Do they tell the same story?


## Tool 1 - Eli5, **global**: and a built-in trap

Eli5 gives you two global importances, and they are **not the same thing**:

- `explain_weights` on a tree model returns the forest's **internal Gini importance** - computed
  from the training splits. It is free, and it is **biased toward high-cardinality features**
  (a continuous feature gets more chances to look useful).
- `PermutationImportance` **shuffles a feature on held-out data** and measures the damage. It is
  model-agnostic, honest about generalisation, and slower.

Most people print the first one and call it importance. Watch what happens when you print both.

In [2]:
import eli5
from eli5.sklearn import PermutationImportance

gini = pd.Series(model.feature_importances_, index=FEATS)

perm = PermutationImportance(model, random_state=SEED, n_iter=10).fit(X_te, y_te)
permimp = pd.Series(perm.feature_importances_, index=FEATS)

cmp = pd.DataFrame({'gini (train, built-in)': gini,
                    'permutation (test, honest)': permimp})
cmp['gini rank'] = cmp['gini (train, built-in)'].rank(ascending=False).astype(int)
cmp['perm rank'] = cmp['permutation (test, honest)'].rank(ascending=False).astype(int)
cmp['rank shift'] = cmp['gini rank'] - cmp['perm rank']

print(cmp.sort_values('perm rank').round(4).to_string())

rho = spearmanr(cmp['gini rank'], cmp['perm rank']).statistic
print(f'\nSpearman rank correlation between the two "importances": {rho:.3f}')
print('Both are labelled "feature importance". They do not agree. Report the one you can defend.')

                      gini (train, built-in)  permutation (test, honest)  gini rank  perm rank  rank shift
alcohol                               0.1764                      0.0492          1          1           0
sulphates                             0.1111                      0.0407          2          2           0
volatile_acidity                      0.1043                      0.0307          3          3           0
density                               0.0930                      0.0197          4          4           0
total_sulfur_dioxide                  0.0806                      0.0195          6          5           1
citric_acid                           0.0893                      0.0180          5          6          -1
residual_sugar                        0.0697                      0.0062          9          7           2
free_sulfur_dioxide                   0.0623                      0.0050         11          8           3
fixed_acidity                        

## Tool 1b - Eli5, **local**: and a landmine

Same library, different scope. For a tree ensemble Eli5 walks the actual decision path and reports
each feature's contribution - a *faithful* decomposition, not an approximation.

Faithful **to which class**, though? Watch carefully.

In [3]:
i = 7
wine = X_te.iloc[[i]]
p = model.predict_proba(wine)[0, 1]
print(f'Wine #{X_te.index[i]} -> P(good) = {p:.3f} '
      f'(actual: {"good" if y_te.iloc[i] else "not good"})\n')

def eli5_weights(target):
    e = eli5.explain_prediction(model, wine.iloc[0], feature_names=FEATS,
                                targets=target, top=len(FEATS)).targets[0]
    fw = e.feature_weights
    w = {x.feature: x.weight for x in list(fw.pos) + list(fw.neg)}
    return e.target, w.get('volatile_acidity')

print('Ask Eli5 to explain each class and watch the weight for volatile_acidity:')
for t in (None, [0], [1]):
    cls, va = eli5_weights(t)
    print(f'  targets={str(t):5s} -> Eli5 says it explained class {cls}, '
          f'volatile_acidity = {va:+.4f}')

print('\nThe class LABEL changes. The WEIGHTS do not. Eli5 hands you the class-0')
print('decomposition and stamps it with whichever class you asked for.')
print('Read as "contribution to P(good)", every sign is inverted. Keep reading.')

Wine #868 -> P(good) = 0.040 (actual: not good)

Ask Eli5 to explain each class and watch the weight for volatile_acidity:
  targets=None  -> Eli5 says it explained class 0, volatile_acidity = +0.0764


  targets=[0]   -> Eli5 says it explained class 0, volatile_acidity = +0.0764
  targets=[1]   -> Eli5 says it explained class 1, volatile_acidity = +0.0764

The class LABEL changes. The WEIGHTS do not. Eli5 hands you the class-0
decomposition and stamps it with whichever class you asked for.
Read as "contribution to P(good)", every sign is inverted. Keep reading.


### Which sign is actually right?

SHAP and Eli5 now contradict each other on `volatile_acidity` - one says it pushed this wine
**toward** good, the other **away**. They cannot both be right, and no chart will tell you which is.

So stop reading explanations and **interrogate the model directly**. Volatile acidity is the vinegar
taint in wine - domain knowledge says more of it is worse. Does the model agree?

In [4]:
print(f"this wine's volatile_acidity = {wine['volatile_acidity'].values[0]:.2f} "
      f"(dataset mean {X['volatile_acidity'].mean():.3f})\n")

print('Ground truth in the data - share of GOOD wines by volatile-acidity quartile:')
print(df.groupby(pd.qcut(df['volatile_acidity'], 4), observed=True)['good']
        .mean().round(3).to_string())

print('\nIntervention - sweep THIS wine\'s volatile_acidity, watch the model:')
for v in [0.2, 0.4, 0.56, 0.8, 1.0]:
    c = wine.copy(); c['volatile_acidity'] = v
    print(f'  volatile_acidity={v:.2f} -> P(good)={model.predict_proba(c)[0,1]:.3f}')

print('\nUnambiguous: high volatile acidity HURTS. Drop this wine to 0.20 and P(good)')
print('goes 0.040 -> 0.407. Its 0.56 is holding it DOWN.\n')
print('  SHAP said  -0.0744  -> pushed AWAY from good.  CORRECT.')
print('  Eli5 said  +0.0764  -> pushed TOWARD good.     WRONG SIGN.')
print('\nEli5 would tell you the vinegar taint is what made the wine good.')
print('It is confident, it is well formatted, and it is backwards.')

this wine's volatile_acidity = 0.56 (dataset mean 0.528)

Ground truth in the data - share of GOOD wines by volatile-acidity quartile:
volatile_acidity
(0.119, 0.39]    0.305
(0.39, 0.52]     0.122
(0.52, 0.64]     0.066
(0.64, 1.58]     0.043

Intervention - sweep THIS wine's volatile_acidity, watch the model:
  volatile_acidity=0.20 -> P(good)=0.407
  volatile_acidity=0.40 -> P(good)=0.343


  volatile_acidity=0.56 -> P(good)=0.040
  volatile_acidity=0.80 -> P(good)=0.050
  volatile_acidity=1.00 -> P(good)=0.077

Unambiguous: high volatile acidity HURTS. Drop this wine to 0.20 and P(good)
goes 0.040 -> 0.407. Its 0.56 is holding it DOWN.

  SHAP said  -0.0744  -> pushed AWAY from good.  CORRECT.
  Eli5 said  +0.0764  -> pushed TOWARD good.     WRONG SIGN.

Eli5 would tell you the vinegar taint is what made the wine good.
It is confident, it is well formatted, and it is backwards.


## Tool 2 - LIME: a **local surrogate**

LIME takes a completely different route. It does not open the model at all. It:

1. **jitters** the instance to make thousands of fake neighbours,
2. asks the black box to label them,
3. fits a **simple linear model** to that local cloud,
4. hands you *that* model's coefficients as the explanation.

So a LIME explanation is a **model of a model**. Which raises the obvious question, and nobody who
ships a LIME chart ever seems to ask it: *is the surrogate any good?* It reports an R² - check it.

In [5]:
from lime.lime_tabular import LimeTabularExplainer

lime_exp = LimeTabularExplainer(
    X_tr.values, feature_names=FEATS, class_names=['not good', 'good'],
    mode='classification', random_state=SEED, discretize_continuous=True)

e = lime_exp.explain_instance(wine.values[0], model.predict_proba, num_features=6)
print(f'LIME local surrogate R2 = {e.score:.3f}   '
      f'(local prediction {e.local_pred[0]:.3f} vs true model {p:.3f})\n')
print('LIME explanation (rule -> weight):')
for rule, w in e.as_list():
    print(f'  {w:+.4f}  {rule}')
print('\nNote LIME explains RULES ("alcohol > 11.10"), not raw features. That readability')
print('is its selling point - and it is why it cannot be compared to SHAP naively.')

LIME local surrogate R2 = 0.512   (local prediction 0.335 vs true model 0.040)

LIME explanation (rule -> weight):
  +0.1680  alcohol > 11.10
  +0.0828  sulphates > 0.73
  +0.0451  density <= 1.00
  -0.0266  0.10 < citric_acid <= 0.26
  -0.0215  0.52 < volatile_acidity <= 0.64
  -0.0200  fixed_acidity <= 7.10

Note LIME explains RULES ("alcohol > 11.10"), not raw features. That readability
is its selling point - and it is why it cannot be compared to SHAP naively.


## Two checks on LIME that nobody runs

**Check 1 - stability.** LIME's neighbours are randomly sampled. Same instance, same model,
different seed: same explanation?

**Check 2 - is the surrogate even fitting?** LIME reports `score` (the local R²) and `local_pred`
(what the *surrogate* predicts). If `local_pred` is nowhere near the black box's actual output, the
surrogate is not approximating the model at that point - and its coefficients are explaining a
**model that does not exist**.

Nobody checks #2. It turns out to be the one that bites.

In [6]:
runs = []
for seed in range(5):
    ex = LimeTabularExplainer(X_tr.values, feature_names=FEATS,
                              class_names=['not good', 'good'], mode='classification',
                              random_state=seed, discretize_continuous=True)
    r = ex.explain_instance(wine.values[0], model.predict_proba, num_features=5)
    top = [FEATS[f] for f, _ in r.as_map()[1]]
    runs.append(top)
    print(f'seed {seed}: R2={r.score:.3f}  local_pred={r.local_pred[0]:.3f}  top-5 = {top}')

same_order = all(r == runs[0] for r in runs)
same_set = all(set(r) == set(runs[0]) for r in runs)
print(f'\nCheck 1 - identical top-5 SET across seeds?   {same_set}')
print(f'Check 1 - identical top-5 ORDER across seeds? {same_order}')
print('  -> Be fair to LIME: the headline feature is stable here. Only the tail reorders.')
print('     Real, but survivable. Now the one that is not.\n')

print(f'Check 2 - the black box says P(good) = {p:.3f}')
print(f"          LIME's surrogate says     = {e.local_pred[0]:.3f}")
print(f'          local R2                  = {e.score:.3f}')
print(f'          absolute error            = {abs(e.local_pred[0] - p):.3f}'
      f'  <-- that is {abs(e.local_pred[0] - p) / max(p, 1e-9):.0f}x the true value')
print('\nThe surrogate does not reproduce the prediction it claims to explain.')
print('Its coefficients are a faithful account of a model that was never deployed.')

seed 0: R2=0.513  local_pred=0.348  top-5 = ['alcohol', 'sulphates', 'density', 'volatile_acidity', 'citric_acid']


seed 1: R2=0.507  local_pred=0.346  top-5 = ['alcohol', 'sulphates', 'density', 'citric_acid', 'volatile_acidity']


seed 2: R2=0.508  local_pred=0.345  top-5 = ['alcohol', 'sulphates', 'density', 'citric_acid', 'volatile_acidity']
seed 3: R2=0.506  local_pred=0.348  top-5 = ['alcohol', 'sulphates', 'density', 'volatile_acidity', 'citric_acid']


seed 4: R2=0.522  local_pred=0.349  top-5 = ['alcohol', 'sulphates', 'density', 'citric_acid', 'volatile_acidity']

Check 1 - identical top-5 SET across seeds?   True
Check 1 - identical top-5 ORDER across seeds? False
  -> Be fair to LIME: the headline feature is stable here. Only the tail reorders.
     Real, but survivable. Now the one that is not.

Check 2 - the black box says P(good) = 0.040
          LIME's surrogate says     = 0.335
          local R2                  = 0.512
          absolute error            = 0.295  <-- that is 7x the true value

The surrogate does not reproduce the prediction it claims to explain.
Its coefficients are a faithful account of a model that was never deployed.


## The showdown: do the explainers **agree**?

Same wine, same model, three explainers. We rank the features by each method's local attribution
and compare. SHAP and Eli5 both decompose the *actual* tree paths, so they ought to line up. LIME
fits a surrogate, so it is free to disagree - and it is the one people show to customers.

In [7]:
import shap

sv = shap.TreeExplainer(model)(X_te)
sv = sv[..., 1] if sv.values.ndim == 3 else sv
shap_local = pd.Series(sv.values[i], index=FEATS)

fw = eli5.explain_prediction(model, wine.iloc[0], feature_names=FEATS,
                             targets=[1], top=len(FEATS)).targets[0].feature_weights
eli5_raw = pd.Series(0.0, index=FEATS)
for w in list(fw.pos) + list(fw.neg):
    if w.feature in eli5_raw.index:
        eli5_raw[w.feature] = w.weight
eli5_fixed = -eli5_raw          # negate: it gave us the class-0 decomposition

lime_local = pd.Series(0.0, index=FEATS)
for fidx, w in e.as_map()[1]:
    lime_local[FEATS[fidx]] = w

board = pd.DataFrame({'SHAP': shap_local,
                      'Eli5 (raw)': eli5_raw,
                      'Eli5 (negated)': eli5_fixed,
                      'LIME': lime_local})
ranks = board.abs().rank(ascending=False)
board['SHAP rank'] = ranks['SHAP'].astype(int)
board['LIME rank'] = ranks['LIME'].astype(int)
print('Local attributions for ONE wine, by three explainers:\n')
print(board.round(4).sort_values('SHAP rank').to_string())

print('\n--- do they agree on the RANKING? (Spearman on |attribution|) ---')
for a, b in [('SHAP', 'Eli5 (raw)'), ('SHAP', 'LIME'), ('Eli5 (raw)', 'LIME')]:
    print(f'  {a:11s} vs {b:11s}: {spearmanr(ranks[a], ranks[b]).statistic:+.3f}')

print('\n--- do they agree on the DIRECTION? (sign agreement with SHAP) ---')
for name, s in [('Eli5 (raw)', eli5_raw), ('Eli5 (negated)', eli5_fixed), ('LIME', lime_local)]:
    nz = (shap_local != 0) & (s != 0)
    agree = (np.sign(shap_local[nz]) == np.sign(s[nz])).mean()
    print(f'  {name:15s}: {agree:.0%} of features point the same way as SHAP')

print('\nRanking agreement hid the problem completely: Eli5 (raw) tracks SHAP at +0.98,')
print('yet points the OPPOSITE WAY on every feature. Rank correlation is not enough.')
print('Negate it and it becomes the best explanation on the board. Nothing told you to.')

Local attributions for ONE wine, by three explainers:

                        SHAP  Eli5 (raw)  Eli5 (negated)    LIME  SHAP rank  LIME rank
volatile_acidity     -0.0744      0.0764         -0.0764 -0.0215          1          5
sulphates             0.0522     -0.0677          0.0677  0.0828          2          2
citric_acid          -0.0479      0.0459         -0.0459 -0.0266          3          4
fixed_acidity        -0.0246      0.0313         -0.0313 -0.0200          4          6
density               0.0171     -0.0106          0.0106  0.0451          5          3
chlorides            -0.0114      0.0194         -0.0194  0.0000          6          9
residual_sugar       -0.0114      0.0072         -0.0072  0.0000          7          9
alcohol               0.0064     -0.0095          0.0095  0.1680          8          1
pH                   -0.0039      0.0070         -0.0070  0.0000          9          9
total_sulfur_dioxide  0.0011     -0.0023          0.0023  0.0000         10

## Tool 3 - Mlxtend: the only one that shows you the **boundary**

Eli5, LIME and SHAP all answer *"how much did each feature contribute?"* - they are **attribution**
methods. Mlxtend answers a different question entirely: *"what shape did the model actually carve
out of the feature space?"*

You can only draw it in 2D, which is the catch. But it makes something visible that no bar chart
can: **where the model is confident, where it is guessing, and how jagged (overfit) it is.**

In [8]:
from mlxtend.plotting import plot_decision_regions
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

pair = ['alcohol', 'volatile_acidity']   # the two features that mattered most
X2_tr = X_tr[pair].values.astype(float)
X2_te = X_te[pair].values.astype(float)

zoo = {
    'LogReg (linear)': make_pipeline(StandardScaler(), LogisticRegression(random_state=SEED)),
    'Tree depth=3 (blocky)': DecisionTreeClassifier(max_depth=3, random_state=SEED),
    'RandomForest (jagged)': RandomForestClassifier(n_estimators=300, random_state=SEED),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (name, clf) in zip(axes, zoo.items()):
    clf.fit(X2_tr, y_tr.values)
    plot_decision_regions(X2_te, y_te.values, clf=clf, legend=0, ax=ax,
                          colors='#3498db,#e74c3c')
    ax.set_xlabel(pair[0]); ax.set_ylabel(pair[1])
    ax.set_title(f'{name}\ntest acc (2 features) = '
                 f'{accuracy_score(y_te, clf.predict(X2_te)):.3f}', fontsize=10)
fig.suptitle('Mlxtend - the decision boundary each model carved (blue = not good, red = good)',
             fontsize=12)
fig.tight_layout()
fig.savefig(OUT / 'a4_decision_regions.png', dpi=130, bbox_inches='tight')
plt.show()
print('Left: a straight line - underfits, but you could defend it in a meeting.')
print('Middle: axis-aligned blocks - literally the if/else rules, drawn.')
print('Right: a jagged coastline - higher accuracy, and islands that are pure memorisation.')

Left: a straight line - underfits, but you could defend it in a meeting.
Middle: axis-aligned blocks - literally the if/else rules, drawn.
Right: a jagged coastline - higher accuracy, and islands that are pure memorisation.


## Discussion (the brief's question)

**"How does having access to these tools compare with the explainer dashboard?"**

The dashboard is a *packaging* decision, not a *capability* one - it is a UI over exactly these
calls. The real difference is that using the libraries directly forces you to see the thing the
dashboard smooths over: **the explainers disagree, and none of them announces it.**

| Tool | Question it answers | Scope | Faithful to the model? | Watch out |
|---|---|---|---|---|
| **Eli5** `explain_weights` | which features did the forest split on? | global | yes (but *training* Gini) | biased to high-cardinality features |
| **Eli5** `PermutationImportance` | what breaks if I shuffle this? | global | yes (held-out) | slow; correlated features share credit |
| **Eli5** `explain_prediction` | what drove *this* one? | local | magnitudes yes, **signs inverted** | returns class-0 weights whatever `targets` says - **negate them** |
| **LIME** | what drove this one, *in plain rules*? | local | **no - it is a surrogate** | **check `score` (R²) and `local_pred`** - here the surrogate missed the prediction by 8x |
| **SHAP** | what drove this one, *additively*? | local + global | yes, with a guarantee | slow on big data |
| **Mlxtend** | what **shape** is the boundary? | global | yes | 2D only |

### The four things this notebook actually proved

1. **Eli5 gave us a confident, backwards explanation - and we caught it.** For a binary classifier,
   `explain_prediction` returns the **class-0** decision-path weights and stamps them with whatever
   class you asked for. `targets=None`, `targets=[0]`, `targets=[1]` all return **the identical
   weights**; only the printed label changes. Read as *"contribution to P(good)"* - which is exactly
   how anyone would read it - **every sign is inverted**. Eli5 reports `volatile_acidity = +0.076`
   (helped this wine be good). The vinegar taint. We verified against the model by intervention:
   drop this wine's volatile acidity 0.56 -> 0.20 and P(good) goes **0.040 -> 0.407**. It was
   *hurting*. **SHAP had the sign right (-0.074); Eli5 had it backwards.**
2. **Rank correlation would never have caught it.** Eli5 (raw) tracks SHAP at **+0.98** on ranking -
   a number that looks like agreement and would sail through any review - while pointing the
   **opposite way on every single feature**. Agreement on *importance* is not agreement on
   *direction*, and only one of those is what you tell a customer.
3. **LIME disagreed too, and its own output said so.** LIME ranks `alcohol` **1st** (+0.168); SHAP
   ranks it **8th of 11** (+0.006). Why: LIME's surrogate predicts **0.335** for a wine the model
   scores at **0.040** - off by ~7x, local R² **0.51**. It never fit the neighbourhood, so its
   coefficients faithfully describe **a model that was never deployed**. That failure was legible
   the whole time, in a number LIME prints and nobody reads.
4. **Attribution is not the whole story.** Mlxtend shows the RandomForest's boundary is a jagged
   coastline with isolated islands - visibly memorised. No bar chart from any other tool contains
   that information.

> **In fairness to LIME:** its top feature was *stable* across 5 seeds. The instability is real but
> confined to the tail, and it is not the damning result. The unfitted surrogate is.

> **In fairness to Eli5:** its *magnitudes* are excellent - negate them and it is the best explainer
> on the board. The library is not broken. It is **silently ambiguous**, which is worse, because a
> broken tool announces itself.

### So what

This is the module's warning arriving with receipts. Aha (Res. 4) says *a wrong explanation is worse
than no explanation*, because it manufactures **unearned trust**. We did not have to construct a
contrived example to show it. We pointed four standard tools at one model and one wine, and got
materially different stories - each rendered with total confidence, none carrying a warning label.

The defence is not a better library - it is a **habit**. Every claim in this notebook that turned out
to be wrong was caught the same way: by **going back to the model and intervening on it**. Change the
feature, re-predict, see which way it moves. That takes three lines and it is the only thing here
that cannot lie to you, because it is not an explanation - it is the model itself.

Then **triangulate** (run two explainers; if they diverge, say so out loud) and **govern** (Sowden,
Res. 5 - someone must be accountable for which explanation ships).

**Carry into Assessment 2:** if you put SHAP in the discussion section, (a) sanity-check the **sign**
of your top feature with a quick intervention sweep, and (b) run permutation importance alongside it.
*"Two independent explainers agree on the top three features, and an intervention confirms the
direction"* is a defensible claim. One bar chart from one library is a liability - as this notebook
just demonstrated on itself.